# Robot Visual Memory — 2. Graph vs caption embedding retrieval

Exactly two pipelines over the same compact gallery. No original dataset download, image
extraction, answer generation or MCP. Start Compose and import the Kaggle export first.
Graph/caption outputs differ, so this compares retrieval pipelines, not a controlled proof that
graph representation alone is superior. The one-episode configuration is exploratory; there is
no held-out environment. Extraction/retrieval settings remain frozen before running the queries.


In [ ]:
# Configuration — use a real exported directory/ZIP after database import.
import os, sys, time
from pathlib import Path
SMOKE = os.getenv('RVM_NOTEBOOK_SMOKE') == '1'
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p/'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError('Launch Jupyter from the project root or notebooks directory')
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT/'src'))
BUNDLE_PATH = Path(os.getenv('RVM_BUNDLE', str(ROOT/'data/robot-visual-memory-compact')))
OUTPUT = Path(os.getenv('RVM_RESULTS', str(ROOT/'results/retrieval')))
LOCAL_SNAPSHOT = None  # optional Path to bundle/model_snapshot, otherwise pinned small HF model download
QRELS_OVERRIDE = None  # optional completed human CSV outside immutable exported bundle
FREEZE_FILE = None  # defaults to BUNDLE/development-freeze.json
CONFIG_OVERRIDE = None  # optional YAML matching the frozen settings from the episode pilot
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Mode:', 'SYNTHETIC SMOKE — NOT BENCHMARK' if SMOKE else 'real retrieval evaluation')


## Validate the compact gallery and annotation status
Bundle checksums, images, dimensions, graph edges and vector-row alignment are checked before
retrieval. Answers stay unopened. Qrels are loaded only by evaluation, never by either retriever.
A separate human CSV can complete an annotation-pending export without modifying its checksums.
Synthetic fixture mode below is only for offline notebook CI and cannot produce real benchmark claims.


In [ ]:
import pandas as pd
from IPython.display import display
from rvm.bundle import open_bundle, validate_bundle
from rvm.io import load_config, read_json, file_hash, digest
from rvm.annotation import load_qrels, annotation_status, validate_qrels
if SMOKE:
    from rvm.fixture import create_single_episode_fixture
    BUNDLE_PATH = create_single_episode_fixture(OUTPUT/'synthetic-fixture', config_path=ROOT/'config.example.yaml')
BUNDLE = open_bundle(BUNDLE_PATH, show_progress=True)
bundle = validate_bundle(BUNDLE, show_progress=True)
if bundle['manifest']['synthetic'] and not SMOKE:
    raise ValueError('Synthetic fixture is forbidden in real evaluation mode')
frames = bundle['frames']
config = load_config(CONFIG_OVERRIDE) if CONFIG_OVERRIDE else read_json(BUNDLE/'selection.json').get('config', load_config(ROOT/'config.example.yaml'))
selected_qids = read_json(BUNDLE/'annotation/query_ids.json')
questions = [q for q in bundle['questions'] if q.query_id in selected_qids]
judgments = load_qrels(QRELS_OVERRIDE) if QRELS_OVERRIDE else bundle['judgments']
validate_qrels(judgments, frames, bundle['questions'])
display(pd.DataFrame(annotation_status(judgments, frames, questions, synthetic=SMOKE)))
evaluation_scope = bundle['manifest'].get('evaluation_scope', 'held_out_environments')
print('Candidate images:', len(frames), 'Selected evaluation questions:', len(questions))
print('Evaluation scope:', evaluation_scope)
if evaluation_scope == 'single_episode_exploratory':
    print('EXPLORATORY: pilot review and retrieval use the same episode; no held-out environment claims')
print('No reference-answer file was loaded')


## Connect the two Python-client retrievers
The graph parser is configurable (`rules` by default, or cached bounded GPT plans). Plans use
the question alone; GPT planning usage/latency is logged separately. The rules vocabulary,
synonyms, partial-match policy and limitations are in README. Graph retrieval never falls back
to vectors. Caption queries use the exact exported BGE contract and Qdrant environment filter.
Model initialization time is measured separately from online query time.


In [ ]:
from rvm.evaluation import verify_frozen
from rvm.normalization import rules_hash
from rvm.planning import RulesPlanner, GPTPlanner
from rvm.retrieval import GraphRetriever, CaptionRetriever
from rvm.embeddings import CaptionEmbedder
model_load_seconds = 0.0
if not SMOKE:
    freeze_hash = verify_frozen(config, FREEZE_FILE or BUNDLE/'development-freeze.json')
    if bundle['manifest']['rules_hash'] != rules_hash():
        raise ValueError('Normalization differs from frozen export')
    from rvm.ingestion import clients, verify_import
    driver, qdrant = clients()
    display(verify_import(BUNDLE, driver, qdrant, config['retrieval']['collection']))
    start = time.perf_counter()
    print('Loading BGE model; Hugging Face displays download progress for uncached files...')
    embedder = CaptionEmbedder(bundle['embedding'], snapshot=LOCAL_SNAPSHOT,
                               cache_folder=ROOT/'artifacts/hf-cache')
    model_load_seconds = time.perf_counter()-start
    parser_name = config['retrieval']['parser']
    if parser_name not in {'rules','gpt'}:
        raise ValueError('Unknown graph parser')
    planner = RulesPlanner() if parser_name == 'rules' else GPTPlanner(OUTPUT/'plan-cache', config['extraction'])
    systems = {'graph': GraphRetriever(driver, planner, config['retrieval']['collection'], config['retrieval']['min_graph_score']),
               'caption_embedding': CaptionRetriever(qdrant, embedder, config['retrieval']['collection'])}
else:
    # Canned results test notebook/report plumbing, not database retrieval or model quality.
    from rvm.schemas import RetrievalResult, Hit
    class SyntheticReplay:
        def __init__(self, reverse=False): self.reverse = reverse
        def retrieve(self, question, environment_id, k):
            ids = [f.frame_id for f in frames if f.environment_id == environment_id]
            if self.reverse: ids.reverse()
            return RetrievalResult(hits=[Hit(frame_id=fid,score=float(len(ids)-i)) for i,fid in enumerate(ids[:k])],
                                   diagnostics={'total_seconds':0.0,'database_seconds':0.0,'error':None,
                                                'synthetic_replay':True})
    systems = {'graph': SyntheticReplay(), 'caption_embedding': SyntheticReplay(True)}
    freeze_hash = 'SYNTHETIC-REPLAY-NOT-BENCHMARK'
print('Retriever checkpoint: exactly', list(systems), 'model load seconds:', model_load_seconds)


## Rankings, metrics and latency
Retrieve up to the environment's full selected gallery size for untruncated MRR; compute
Precision/Recall/nDCG at k=1,3,5. P@k always divides by k, penalizing missing positions even
when the gallery is smaller than k. Recall divides by the total number of relevant gallery
images. Binary nDCG uses log2(rank+1) discounts. Duplicate/wrong-environment results fail validation.

Primary means use the **same fully human-judged, evidence-bearing queries** for both methods.
No-relevant queries have P=0 and undefined Recall/MRR/nDCG and are reported as coverage cases.
Partial/pooled/silver-only galleries have null metrics: unjudged is never treated as 0.
Retrieval errors yield empty results and are penalized in eligible primary metrics, with errors reported.

First pass and warm passes include query embedding or planning; DB-only times are separate.
First pass is a practical query/cache proxy: Neo4j/Qdrant/OS caches are not flushed, and BGE weights
have already loaded. For actual cold-start experiments restart services and kernel, record startup,
and run once. Do not claim these proxy times are fully cold database latencies.


In [ ]:
from rvm.evaluation import evaluate, report_results
from rvm.extraction import usage_summary, usage_delta
planning_usage_before = usage_summary(OUTPUT/'plan-cache')
table, runs, status = evaluate(systems, questions, frames, judgments,
                               k_values=config['retrieval']['k_values'],
                               warm_repeats=config['retrieval']['warm_repeats'], synthetic=SMOKE,
                               show_progress=True)
aggregate, by_category, diagnostics, paired = report_results(table, runs, status, OUTPUT, provenance={
    'synthetic': SMOKE, 'evaluation_scope': evaluation_scope, 'freeze_hash': freeze_hash, 'config_hash': digest(config),
    'bundle_manifest_hash': file_hash(BUNDLE/'bundle.json'), 'bundle_checksums_hash': file_hash(BUNDLE/'checksums.json'),
    'qrels_hash': file_hash(QRELS_OVERRIDE or BUNDLE/'annotation/qrels.csv'),
    'versions': bundle['manifest']['versions'], 'model_load_seconds': model_load_seconds,
    'online_query_usage': 'See runs.json and plan-cache attempts; rules/BGE have no hosted embedding cost',
    'online_planning_usage': usage_delta(planning_usage_before, usage_summary(OUTPUT/'plan-cache')),
    'offline_extraction_usage': read_json(BUNDLE/'provenance/extraction-usage.json')
        if (BUNDLE/'provenance/extraction-usage.json').exists() else None,
})
if any(not s['complete'] for s in status):
    print('INCOMPLETE EVALUATION: complete human judgments before full-gallery ground-truth claims')
if SMOKE: print('SYNTHETIC REPLAY — all displayed numbers are test-only')
display(aggregate, by_category, diagnostics, paired, table)
latency = pd.read_csv(OUTPUT/'latency.csv')
display(latency.groupby(['method','phase']).total_seconds.agg(['count','mean','median','max']))
print('Results exported:', OUTPUT)


## Side-by-side evidence and failure review
Scores are pipeline-specific; compare ranked evidence and metrics, never raw score magnitudes.
Graph explanations report matched plan constraints, not truth labels. Record failure causes such
as sampled-gallery coverage, caption omission, object/state extraction error, relation direction,
synonym/parser gaps, or distractors in a separate review file; do not relabel from model outputs.


In [ ]:
from rvm.evaluation import compare_display
for query in questions[:3]:
    compare_display(query, runs, frames, BUNDLE, k=max(config['retrieval']['k_values']))
display(table.groupby(['method','failure_category']).size().to_frame('query-k cases'))
if not SMOKE:
    driver.close()
    qdrant.close()
print('Evaluation finished; consult report.json for annotation completeness and provenance')
